# TradePulse - Steps 9 & 10: Diagnostics, Probability Calibration & Confidence Gating
**IIT Patna, IMPACT Lab | End-to-End Machine Learning Guidelines**  
**Asset:** META (NASDAQ) | **Timeframe:** 5-minute candles | **Prediction Horizon:** 15-minute (3 bars ahead)  
**Frozen Target Label:** `label_fixed_20bp` ($\theta = \pm 0.20\%$)  

---
### Objectives & Strict Non-Negotiables
1. **Diagnostics:** Test whether models learn actual direction or merely intraday volatility timing (Time-of-Day baseline, Direction-only accuracy with 95% CIs, Move vs No-Move AUC, Up vs Down AUC, paired day-level block bootstrap).
2. **Step 9 (Calibration):** Generate out-of-fold predictions on Training (5-fold Purged Walk-Forward CV) and fit Platt Sigmoid and Isotonic calibrators. Evaluate Brier score, log loss, and reliability curves on the Validation set.
3. **Step 10 (Confidence Gating):** Evaluate a predefined grid of probability thresholds ($\tau \in [0.35, 0.60]$) and quantile cutoffs with day-level block-bootstrap 95% CIs. Select threshold on Training OOF and confirm on Validation.
4. **Realistic Backtest:** Non-overlapping 15-minute execution (positions block new entries for 3 bars) under 5, 8, 12 bps transaction costs.
5. **Honest Reporting:** Acknowledge whether any profitable net edge exists after transaction costs.
6. **Zero-Leakage Constraint:** The Test Set (`test_features.csv`) remains completely sealed and untouched.


In [ ]:
import sys
import json
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

with open(PROJECT_ROOT / "artifacts" / "metadata" / "step9_step10_diagnostics_and_gating.json", "r", encoding="utf-8") as f:
    step_data = json.load(f)

print("Diagnostic and gating artifacts loaded successfully.")


## 1. Part 1: Diagnostic Suite
Checking whether the models predict true price direction or merely capture time-of-day volatility clustering.


In [ ]:
df_diag = pd.DataFrame(step_data["diagnostics"])
print("=== Model Diagnostics Comparison on Validation Set ===")
display(df_diag)

print("\n--- Paired Day-Level Block Bootstrap Results (B = 1,000 resamples) ---")
for comp_name, b_res in step_data["bootstrap_model_comparisons"].items():
    print(f"{comp_name:<25}: Diff Gross Return = {b_res['diff_gross_return_bps_mean']:+.2f} bps, 95% CI {b_res['diff_gross_return_bps_95ci']} | Distinguishable from Zero: {b_res['gross_return_distinguishable_from_zero']}")


### Performance Breakdown by 30-Minute Time-of-Day Slots


In [ ]:
df_tod = pd.DataFrame(step_data["performance_by_tod"])
display(df_tod[["slot_time", "coverage_pct", "precision_pct", "gross_return_bps", "net_return_8bp_bps", "win_rate_pct"]])

fig_path = PROJECT_ROOT / "reports" / "figures" / "step9_performance_by_time_of_day.png"
if fig_path.exists():
    display(Image(filename=str(fig_path)))


## 2. Part 2: Step 9 Probability Calibration
Evaluating Platt Sigmoid vs. Isotonic calibration (fitted strictly on 5-fold training out-of-fold predictions).


In [ ]:
df_cal = pd.DataFrame(step_data["calibration_comparison"])
print("=== Calibration Quality Comparison (Validation Set) ===")
display(df_cal)

fig_rel = PROJECT_ROOT / "reports" / "figures" / "step9_reliability_curves.png"
if fig_rel.exists():
    display(Image(filename=str(fig_rel)))

fig_loss = PROJECT_ROOT / "reports" / "figures" / "step9_calibration_loss_comparison.png"
if fig_loss.exists():
    display(Image(filename=str(fig_loss)))


## 3. Part 3: Step 10 Confidence Gating & Threshold Search
Evaluating the predefined grid on Training Out-of-Fold predictions and confirming on the Validation set.


In [ ]:
cols_grid = ["rule_desc", "effective_threshold", "coverage_pct", "signals_per_day", "zero_signal_days_pct", "precision_pct", "direction_accuracy_pct", "gross_return_bps", "net_8bp_bps", "net_8bp_95ci", "profitable_edge_at_8bp"]

print("=== Training Out-of-Fold Gating Grid (Random Forest Calibrated) ===")
df_oof_grid = pd.DataFrame(step_data["training_oof_gating_grid"])
display(df_oof_grid[cols_grid])

print("\n=== Validation Gating Grid (Random Forest Calibrated - Confirmation Only) ===")
df_val_grid = pd.DataFrame(step_data["validation_gating_grid_rf"])
display(df_val_grid[cols_grid])


## 4. Part 4: Realistic Non-Overlapping Backtest Simulation
Simulating 15-minute holding periods where active positions block new entries for 3 bars under 8 bps friction.


In [ ]:
df_sim = pd.DataFrame(step_data["validation_simulation_non_overlapping"])
print("=== Non-Overlapping Backtest Summary (Validation Set, 42 Sessions) ===")
display(df_sim)

fig_equity = PROJECT_ROOT / "reports" / "figures" / "step10_non_overlapping_equity_curves.png"
if fig_equity.exists():
    display(Image(filename=str(fig_equity)))


## 5. Conclusions & Production Rule Alignment
1. **Move vs. Direction:** Move vs. No-Move AUC is ~0.67, but Up vs. Down AUC is ~0.50. The model effectively detects volatility expansion regimes, but has near-zero directional edge over short 15-minute windows.
2. **Calibration:** Platt sigmoid scaling successfully reduces Random Forest log loss from 1.0679 to 0.9356, yielding trustworthy probability outputs.
3. **Economic Result:** No subset of gated signals achieves a statistically verifiable positive net return after 8 bps friction. The lower bound of the 95% day-level bootstrap CI remains negative across all cutoffs.
4. **Conservative Production Configuration:** In accordance with IIT Patna IMPACT Lab guidelines, the indicator ships as a conservative tool configured to output 'No Clear Signal' unless high conviction is present.
